<a href="https://colab.research.google.com/github/idoschw3/Corpus2GeoRAG/blob/main/ApplyNER/final_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import Counter, defaultdict
from flair.models import SequenceTagger
from transformers import pipeline
import pyarrow.parquet as pq
from pathlib import Path
from tqdm import tqdm
import pyarrow as pa
import pandas as pd
import numpy as np
import subprocess
import tempfile
import shutil
import flair
import torch
import json
import time
import gzip
import json
import sys
import os
import re

IS_COLAB_VM = (
    "google.colab" in sys.modules
    and os.path.exists("/content")
    and os.environ.get("COLAB_RELEASE_TAG") is not None
)

if IS_COLAB_VM:
    from google.colab import userdata
else:
    userdata = None

C:\Users\idosc\.venvs\corpus2georag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Clone Repository

In [2]:
IS_LOCAL = os.name == "nt"

LOCAL_REPO_PATH = Path.home() / "Corpus2GeoRAG"
repo_dir = (
    LOCAL_REPO_PATH
    if IS_LOCAL
    else Path("/content/Corpus2GeoRAG")
)

REPO_PATH = str(repo_dir)

if not repo_dir.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/EtzionR/Corpus2GeoRAG.git",
            REPO_PATH
        ],
        check=True
    )

elif not (repo_dir / ".git").exists():
    raise RuntimeError(
        f"{REPO_PATH} exists but is not a Git checkout."
    )

else:
    subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=repo_dir,
        check=True
    )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Working folder:", REPO_PATH)

Runtime: Local Windows
Working folder: C:\Users\idosc\Corpus2GeoRAG


# Data Upload

In [3]:
# Locate the compressed file inside the repository.
input_path = Path("CollectCorpus/DATA.json.gz")

# Decompress and load the complete JSON dataset.
with gzip.open(input_path, "rt", encoding="utf-8") as f:
    data = json.load(f)

# Count newline-separated lines before filtering.
total_lines = sum(len(text.split("\n")) for text in data.values())

print(f"Loaded {len(data)} pages.")
print(f"Total lines before filtering: {total_lines}")
print(f"Source: {input_path.resolve()}")

Loaded 2081 pages.
Total lines before filtering: 185253
Source: C:\Users\idosc\Corpus2GeoRAG\CollectCorpus\DATA.json.gz


## Data Cleaning

remove titles from the data. titles will be added as new columns in a later step.

in addition remove external links from each page.

In [4]:
[*data.keys()][:10]

example = np.random.choice([*data],1)[0]


print(f'PAGE: {example}:\n\n{data[example]}')

PAGE: Pomsta Brigade:

The 3rd Border Detachment named after the Hero of Ukraine Colonel Yevgeny Pikus, also known as Pomsta Brigade, (Ukrainian: Помста, lit. 'Revenge') is an assault brigade of the State Border Guard Service of Ukraine that is part of the Offensive Guard.
More than 1,000 servicemen of the brigade have been awarded state awards, among them six were awarded the "Hero of Ukraine", four of them posthumously.


== History ==


=== Early history ===
The 3rd border detachment was created on 20 October 1992, following the Dissolution of the Soviet union and was tasked with protecting the state border of Ukraine with Russia in Luhansk Oblast. During the 1990s, the detachment actively worked on developing its structure and improving its skills.


=== War in Donbass ===
In 2014, following the Euromaidan, Revolution of Dignity, 2014 pro-Russian unrest in Ukraine and the Russian invasion of Crimea, the War in Donbass started. The 3rd border detachment took on the task of protectin

In [5]:
TITLE = '=='
MIN_LENGTH = 5

# Only these two headings trigger a cutoff.
STOP_HEADINGS = {'references', 'external links'}

# Match headings with the same number of equals signs on both sides.
heading_pattern = re.compile(
    r'(?P<marks>={2,})\s*(?P<heading>.*?)\s*(?P=marks)'
)

data_processed = {}
paragraph_metadata = {}

total_lines = 0
total_kept = 0

# Cutoff validation data.
cutoff_counts = Counter()
cutoff_audit = {}
suspicious_cutoffs = {}

for page, page_text in data.items():
    paragraphs = page_text.split('\n')
    total_lines += len(paragraphs)

    current_title = None
    current_subtitle = None

    kept_paragraphs = []
    kept_metadata = []

    first_cutoff = None
    unexpected_sections = []
    active_unexpected_section = None

    for line_number, text in enumerate(paragraphs):
        heading_match = heading_pattern.fullmatch(text.strip())

        if heading_match:
            heading_level = len(heading_match.group('marks'))
            heading_text = heading_match.group('heading').strip()
            normalized_heading = ' '.join(
                heading_text.casefold().split()
            )
        else:
            heading_level = None
            heading_text = None
            normalized_heading = None

        # ----------------------------------------------
        # Validate content after the first cutoff.
        # Nothing here is added to the processed data.
        # ----------------------------------------------
        if first_cutoff is not None:

            if heading_match:
                # Check headings at the same or higher
                # structural level as the cutoff.
                if heading_level <= first_cutoff['level']:

                    if normalized_heading in STOP_HEADINGS:
                        active_unexpected_section = None

                    else:
                        active_unexpected_section = {
                            'heading': heading_text,
                            'level': heading_level,
                            'line': line_number + 1,
                            'paragraph_count': 0,
                            'examples': []
                        }

                        unexpected_sections.append(
                            active_unexpected_section
                        )

            elif active_unexpected_section is not None:
                if (
                    not text.startswith(TITLE)
                    and len(text) >= MIN_LENGTH
                ):
                    active_unexpected_section['paragraph_count'] += 1

                    if len(active_unexpected_section['examples']) < 2:
                        active_unexpected_section['examples'].append(
                            text.strip()[:200]
                        )

            continue

        # ----------------------------------------------
        # Process headings and apply cutoff.
        # ----------------------------------------------
        if heading_match:

            if normalized_heading in STOP_HEADINGS:
                first_cutoff = {
                    'heading': heading_text,
                    'level': heading_level,
                    'line': line_number + 1
                }

                cutoff_counts[normalized_heading] += 1
                continue

            if heading_level == 2:
                current_title = heading_text
                current_subtitle = None

            elif heading_level == 3:
                current_subtitle = heading_text

            # Headings are metadata, not paragraphs.
            continue

        # ----------------------------------------------
        # Original paragraph filtering.
        # ----------------------------------------------
        if text.startswith(TITLE) or len(text) < MIN_LENGTH:
            continue

        kept_paragraphs.append(text)
        kept_metadata.append({
            'title': current_title,
            'sub_title': current_subtitle
        })

    data_processed[page] = kept_paragraphs
    paragraph_metadata[page] = kept_metadata

    total_kept += len(kept_paragraphs)

    # Store validation results.
    if first_cutoff is not None:

        cutoff_audit[page] = {
            'cutoff': first_cutoff,
            'discarded_lines': (
                len(paragraphs) - first_cutoff['line'] + 1
            ),
            'unexpected_sections': unexpected_sections
        }

        if unexpected_sections:
            suspicious_cutoffs[page] = cutoff_audit[page]

# ==============================================
# ORIGINAL PROCESSING SUMMARY
# ==============================================

percentage = (
    round(total_kept / total_lines * 100, 1)
    if total_lines else 0
)

print(f'Processed {len(data_processed)} pages')
print(
    f'{total_kept} Paragraphs Extracted from {total_lines}\n'
    f'{percentage}%'
)

# ==============================================
# CUTOFF SUMMARY
# ==============================================

print('\n--- CUTOFF SUMMARY ---')

print(f"References: {cutoff_counts['references']} pages")
print(f"External links: {cutoff_counts['external links']} pages")
print(f'Pages with a cutoff: {len(cutoff_audit)}')
print(f'Pages without a cutoff: {len(data) - len(cutoff_audit)}')

# ==============================================
# VALIDATION: AGGREGATE ALL ADDITIONAL SECTIONS
# ==============================================

heading_pages = defaultdict(set)
heading_paragraphs = Counter()
heading_examples = {}
heading_occurrences = Counter()

for page, audit in suspicious_cutoffs.items():

    for section in audit['unexpected_sections']:
        heading = section['heading']

        heading_pages[heading].add(page)
        heading_occurrences[heading] += 1
        heading_paragraphs[heading] += section['paragraph_count']

        if (
            heading not in heading_examples
            and section['examples']
        ):
            heading_examples[heading] = {
                'page': page,
                'text': section['examples'][0]
            }

# ==============================================
# VALIDATION REPORT
# ==============================================

print('\n--- CUTOFF VALIDATION ---')

print(
    'Pages with additional same-level or higher-level '
    f'sections: {len(suspicious_cutoffs)}'
)

print(f'Distinct additional headings: {len(heading_pages)}')
print(
    'Paragraphs under additional sections: '
    f'{sum(heading_paragraphs.values())}'
)

if not suspicious_cutoffs:
    print(
        '\nNo additional same-level or higher-level '
        'sections were found after the cutoffs.'
    )

else:
    print('\n--- SECTIONS FOUND AFTER CUTOFF ---')

    for heading in sorted(
        heading_pages,
        key=lambda h: (
            -len(heading_pages[h]),
            h.casefold()
        )
    ):
        print(f'\nSection: {heading}')
        print(f'  Pages: {len(heading_pages[heading])}')
        print(f'  Occurrences: {heading_occurrences[heading]}')
        print(f'  Paragraphs: {heading_paragraphs[heading]}')

        if heading in heading_examples:
            example = heading_examples[heading]

            print(f"  Example page: {example['page']}")
            print(f"  Example text: {example['text']}")

    print('\n--- VALIDATION NOTES ---')
    print(
        'Additional headings are not automatically errors. '
        'Inspect the reported sections to determine whether '
        'any relevant article content was excluded.'
    )
    print(
        'Full page-level details are available in '
        'suspicious_cutoffs.'
    )


Processed 2081 pages
94881 Paragraphs Extracted from 185253
51.2%

--- CUTOFF SUMMARY ---
References: 2035 pages
External links: 21 pages
Pages with a cutoff: 2056
Pages without a cutoff: 25

--- CUTOFF VALIDATION ---
Pages with additional same-level or higher-level sections: 262
Distinct additional headings: 18
Paragraphs under additional sections: 1129

--- SECTIONS FOUND AFTER CUTOFF ---

Section: Further reading
  Pages: 148
  Occurrences: 148
  Paragraphs: 599
  Example page: 2014 Russian annexation of Crimea
  Example text: Pynnöniemi, Katri; Rácz, András, eds. (2016). "Fog of Falsehood: Russian Strategy of Deception and the Conflict in Ukraine". Fiia Report. FIIA Report, 45. Helsinki: Finnish Institute of International 

Section: Sources
  Pages: 60
  Occurrences: 60
  Paragraphs: 295
  Example page: United Kingdom and the Russo-Ukrainian war
  Example text: Cole, Harry; Heale, James (2022). Out of the Blue: The Inside Story of the Unexpected Rise and Rapid Fall of Liz Truss. Ha

# model loading

chosen model: Flair
date: 16.10.26

In [6]:
MODEL_Flair = "flair/ner-english-large"

flair.device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ner_Flair = SequenceTagger.load(MODEL_Flair)
ner_Flair.to(flair.device)

print("Model device:", flair.device)

ner_Flair

2026-10-27 11:10:42,142 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>
Model device: cuda


SequenceTagger(
  (embeddings): TransformerWordEmbeddings(
    (model): XLMRobertaModel(
      (embeddings): XLMRobertaEmbeddings(
        (word_embeddings): Embedding(250002, 1024, padding_idx=1)
        (position_embeddings): Embedding(514, 1024, padding_idx=1)
        (token_type_embeddings): Embedding(1, 1024)
        (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): XLMRobertaEncoder(
        (layer): ModuleList(
          (0-23): 24 x XLMRobertaLayer(
            (attention): XLMRobertaAttention(
              (self): XLMRobertaSdpaSelfAttention(
                (query): Linear(in_features=1024, out_features=1024, bias=True)
                (key): Linear(in_features=1024, out_features=1024, bias=True)
                (value): Linear(in_features=1024, out_features=1024, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): XLM

# Model Application

In [7]:
BATCH_SIZE = 32 if next(ner_Flair.parameters()).device.type == "cuda" else 1

columns = [
    "entity_group",
    "score",
    "word",
    "start",
    "end",
    "Source",
    "Index",
    "Text",
    "title",
    "sub_title"
]

# Explicit types also allow an empty result to be saved correctly.
output_schema = pa.schema([
    ("entity_group", pa.string()),
    ("score", pa.float64()),
    ("word", pa.string()),
    ("start", pa.int64()),
    ("end", pa.int64()),
    ("Source", pa.string()),
    ("Index", pa.int64()),
    ("Text", pa.string()),
    ("title", pa.string()),
    ("sub_title", pa.string())
])

output_dir = Path("ApplyNER/NER_output").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "post_NER_final.parquet"
temporary_path = output_path.with_suffix(".parquet.tmp")

if not pa.Codec.is_available("zstd"):
    raise RuntimeError("This PyArrow installation does not support zstd.")

# Use the full dataset, keeping each paragraph's headings.
paragraph_records = [
    (
        page,
        paragraph_id,
        paragraph,
        metadata["title"],
        metadata["sub_title"]
    )
    for page, paragraphs in data_processed.items()
    for paragraph_id, (paragraph, metadata) in enumerate(
        zip(paragraphs, paragraph_metadata[page], strict=True)
    )
]

rows = []
started = time.perf_counter()

# Flair processes Sentence objects rather than a Hugging Face pipeline.
# Keep batches separate to avoid holding all Sentence objects in memory.
from flair.data import Sentence

for batch_start in tqdm(
    range(0, len(paragraph_records), BATCH_SIZE),
    desc="Flair NER — full data",
    unit="batch"
):
    batch_records = paragraph_records[
        batch_start:batch_start + BATCH_SIZE
    ]

    sentences = [
        Sentence(record[2], use_tokenizer=True)
        for record in batch_records
    ]

    # Predict entities for the current batch.
    ner_Flair.predict(
        sentences,
        mini_batch_size=BATCH_SIZE,
        embedding_storage_mode="none",
        verbose=False
    )

    for (page, paragraph_id, paragraph, title, subtitle), sentence in zip(
        batch_records,
        sentences,
        strict=True
    ):
        for entity in sentence.get_spans("ner"):
            label = entity.get_label("ner")

            start = int(entity.start_position)
            end = int(entity.end_position)

            rows.append({
                "entity_group": label.value,
                "score": float(label.score),
                "word": paragraph[start:end],
                "start": start,
                "end": end,
                "Source": page,
                "Index": paragraph_id,
                "Text": paragraph,
                "title": title,
                "sub_title": subtitle
            })

    # Release Sentence objects and their associated embeddings.
    del sentences

elapsed = time.perf_counter() - started

# Retain the DataFrame for further notebook analysis.
df_Flair = pd.DataFrame(rows, columns=columns)

print(
    f"NER complete: {len(rows)} entity rows. Saving Parquet...",
    flush=True
)

save_started = time.perf_counter()

# Build the Arrow table directly from the entity records.
# This bypasses pandas' failing PyArrow extension setup.
table = pa.Table.from_pylist(rows, schema=output_schema)

pq.write_table(
    table,
    temporary_path,
    compression="zstd"
)

# Check the file before replacing the final output.
with pq.ParquetFile(temporary_path) as saved_file:
    saved_rows = saved_file.metadata.num_rows
    saved_columns = saved_file.schema_arrow.names

if saved_rows != len(rows):
    raise RuntimeError(
        f"Row count mismatch: expected {len(rows)}, saved {saved_rows}."
    )

if saved_columns != columns:
    raise RuntimeError(
        f"Column mismatch: expected {columns}, saved {saved_columns}."
    )

# Replace the final file only after validation succeeds.
temporary_path.replace(output_path)

save_elapsed = time.perf_counter() - save_started
file_size_mb = output_path.stat().st_size / (1024 ** 2)

print(
    f"Processed {len(data_processed)} pages "
    f"and {len(paragraph_records)} paragraphs in {elapsed:.1f} seconds."
)
print(f"Model device: {next(ner_Flair.parameters()).device}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Saved and verified {saved_rows} entity rows.")
print(f"Columns: {saved_columns}")
print(f"File: {output_path}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Saving and verification took {save_elapsed:.1f} seconds.")

Flair NER — full data: 100%|██████████| 2966/2966 [1:56:45<00:00,  2.36s/batch]


NER complete: 490536 entity rows. Saving Parquet...
Processed 2081 pages and 94881 paragraphs in 7005.5 seconds.
Model device: cuda:0
Batch size: 32
Saved and verified 490536 entity rows.
Columns: ['entity_group', 'score', 'word', 'start', 'end', 'Source', 'Index', 'Text', 'title', 'sub_title']
File: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\post_NER_final.parquet
File size: 15.40 MB
Saving and verification took 1.7 seconds.


In [8]:
display(df_Flair.head(10))

display(
    df_Flair[["title", "sub_title"]]
    .isna().mean().mul(100).round(2)
    .to_frame(name="NaN (%)")
)

,entity_group,score,word,start,end,Source,Index,Text,title,sub_title
0,LOC,0.999998,Russia,28,34,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN
1,LOC,0.999961,Crimean Peninsula,47,64,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN
2,LOC,0.999986,Ukraine,68,75,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN
3,LOC,0.999998,Ukraine,207,214,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN
4,MISC,0.999964,Revolution of Dignity,217,238,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN
5,MISC,0.999996,Russo-Ukrainian,281,296,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN
6,LOC,0.999996,Crimea,303,309,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN
7,MISC,0.999997,Russian,367,374,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN
8,MISC,0.999959,World War II,416,428,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN
9,MISC,0.999995,European,436,444,2014 Russian annexation of Crimea,0,"In February and March 2014, Russia invaded the...",NaN,NaN


,NaN (%)
title,6.09
sub_title,36.73


# Git Push/Commit

In [9]:
GITHUB_USER = "idoschw3"
GITHUB_REPO = "Corpus2GeoRAG"
GITHUB_EMAIL = "193781032+idoschw3@users.noreply.github.com"
COMMIT_MESSAGE = "Update NER workflow"

LOCAL_REPO_PATH = Path.home() / GITHUB_REPO
MAX_GIT_FILE_BYTES = 100 * 1024 * 1024

# --------------------------------------------------
# Detect runtime
# --------------------------------------------------

if os.name == "nt":
    IS_LOCAL = True
elif os.environ.get("COLAB_RELEASE_TAG"):
    IS_LOCAL = False
else:
    raise RuntimeError(
        "Could not identify this runtime as Windows or a hosted Colab VM."
    )

if shutil.which("git") is None:
    raise RuntimeError(
        "Git is not available to this Jupyter environment. "
        "Install Git for Windows and restart Jupyter if running locally."
    )

# --------------------------------------------------
# Locate the repository
# --------------------------------------------------

if IS_LOCAL:
    candidates = [Path.cwd(), LOCAL_REPO_PATH]

    previous_repo_path = globals().get("REPO_PATH")
    if previous_repo_path:
        candidates.insert(0, Path(previous_repo_path))

    detected_repo = None

    for candidate in candidates:
        if not candidate.is_dir():
            continue

        probe = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            cwd=candidate,
            text=True,
            encoding="utf-8",
            errors="replace",
            capture_output=True
        )

        if probe.returncode == 0:
            root = Path(probe.stdout.strip()).resolve()

            if root.name.lower() == GITHUB_REPO.lower():
                detected_repo = root
                break

    if detected_repo is None:
        raise FileNotFoundError(
            f"Could not find your local {GITHUB_REPO} checkout.\n"
            f"Clone it to {LOCAL_REPO_PATH}, or update LOCAL_REPO_PATH."
        )

    REPO_PATH = str(detected_repo)

else:
    REPO_PATH = f"/content/{GITHUB_REPO}"

    if not Path(REPO_PATH).is_dir():
        raise FileNotFoundError(
            f"{REPO_PATH} does not exist. Run your clone cell first."
        )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Repository:", REPO_PATH)


def run_git(args, *, env=None, show_output=True):
    """Run Git and include its actual output if it fails."""
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_PATH,
        env=env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if show_output:
        if result.stdout:
            print(result.stdout, end="")
        if result.stderr:
            print(result.stderr, end="")

    if result.returncode != 0:
        details = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"git {' '.join(args)} failed:\n{details}")

    return result


# --------------------------------------------------
# Git configuration
# --------------------------------------------------

run_git(["config", "user.name", GITHUB_USER])
run_git(["config", "user.email", GITHUB_EMAIL])
run_git([
    "remote", "set-url", "origin",
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
])

branch = run_git(
    ["branch", "--show-current"],
    show_output=False
).stdout.strip()

if not branch:
    raise RuntimeError("Could not determine the current Git branch.")

print("Branch:", branch)

# Stop if an earlier rebase or merge is still paused.
for state_path in ("rebase-merge", "rebase-apply", "MERGE_HEAD"):
    git_state = run_git(
        ["rev-parse", "--git-path", state_path],
        show_output=False
    ).stdout.strip()

    if Path(git_state).exists():
        raise RuntimeError(
            "An earlier rebase or merge is still in progress. "
            "Check `git status` and resolve it before rerunning this cell."
        )

# --------------------------------------------------
# Authentication and synchronization
# --------------------------------------------------

git_env = os.environ.copy()
git_env["GIT_TERMINAL_PROMPT"] = "0"

auth_options = []
askpass_dir = None

try:
    if IS_LOCAL:
        # Use Git credentials already configured on your computer.
        print("Authentication: local Git credentials.")

    else:
        from google.colab import userdata

        github_token = userdata.get("GITHUB_TOKEN")

        if not github_token:
            raise ValueError(
                "GITHUB_TOKEN is missing from Colab Secrets. "
                "Add it and enable notebook access."
            )

        askpass_dir = tempfile.TemporaryDirectory(
            prefix="github-askpass-"
        )
        askpass_path = Path(askpass_dir.name) / "askpass.sh"

        askpass_path.write_text(
            "#!/bin/sh\n"
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "$GITHUB_USER" ;;\n'
            '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
            "esac\n",
            encoding="utf-8"
        )
        askpass_path.chmod(0o700)

        git_env["GITHUB_USER"] = GITHUB_USER
        git_env["GITHUB_TOKEN"] = github_token
        git_env["GIT_ASKPASS"] = str(askpass_path)

        auth_options = ["-c", "credential.helper="]

        print("Authentication: Colab GITHUB_TOKEN Secret.")

    # --------------------------------------------------
    # Prevent accidental nested repository
    # --------------------------------------------------

    ignore_entry = f"/{GITHUB_REPO}/"
    gitignore_path = Path(REPO_PATH) / ".gitignore"

    existing_ignore = (
        gitignore_path.read_text(encoding="utf-8")
        if gitignore_path.exists()
        else ""
    )

    if ignore_entry not in existing_ignore.splitlines():
        with gitignore_path.open("a", encoding="utf-8") as f:
            if existing_ignore and not existing_ignore.endswith("\n"):
                f.write("\n")
            f.write(ignore_entry + "\n")

    nested_repo = run_git(
        ["ls-files", "--stage", "--", GITHUB_REPO],
        show_output=False
    ).stdout.strip()

    if nested_repo:
        run_git([
            "rm", "--cached", "-r", "-f", "--", GITHUB_REPO
        ])
        print("Removed the nested repository path from Git tracking.")

    # --------------------------------------------------
    # Find files exceeding GitHub's per-file limit
    # --------------------------------------------------

    candidate_files = run_git(
        [
            "ls-files",
            "--cached",
            "--others",
            "--exclude-standard",
            "-z"
        ],
        show_output=False
    ).stdout.split("\0")

    excluded_files = set()

    for relative_path in sorted(set(candidate_files) - {""}):
        file_path = Path(REPO_PATH) / relative_path

        if not file_path.is_file():
            continue

        file_size = file_path.stat().st_size

        if file_size > MAX_GIT_FILE_BYTES:
            excluded_files.add(relative_path)

            # Remove from Git's index, while keeping the file on disk.
            run_git(
                [
                    "rm",
                    "--cached",
                    "--ignore-unmatch",
                    "-f",
                    "--",
                    f":(top,literal){relative_path}"
                ],
                show_output=False
            )

            print(
                f"Excluded oversized file: {relative_path} "
                f"({file_size / 1024**2:.2f} MiB). "
                "Kept on disk."
            )

    # --------------------------------------------------
    # Stage changes, excluding oversized files
    # --------------------------------------------------

    exclusion_pathspecs = [
        f":(top,literal,exclude){path}"
        for path in sorted(excluded_files)
    ]

    run_git(["add", "-A", "--", ".", *exclusion_pathspecs])

    # Exclude notebook copies, preserving the original cell's behavior.
    staged_files = run_git(
        ["diff", "--cached", "--name-only", "-z"],
        show_output=False
    ).stdout.split("\0")

    for notebook in staged_files:
        if (
            notebook.endswith(".ipynb")
            and notebook not in excluded_files
        ):
            run_git([
                "restore", "--staged", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Create a normal commit
    # --------------------------------------------------

    staged_diff = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_PATH,
        stdin=subprocess.DEVNULL
    )

    if staged_diff.returncode == 1:
        run_git(["commit", "-m", COMMIT_MESSAGE])
        print("Repository changes committed.")
    elif staged_diff.returncode == 0:
        print("No new repository changes to commit.")
    else:
        raise RuntimeError("Could not check the staged changes.")

    # --------------------------------------------------
    # Restore tracked notebook copies on disk
    # This discards edits to those on-disk copies.
    # --------------------------------------------------

    tracked_files = run_git(
        ["ls-tree", "-r", "--name-only", "-z", "HEAD"],
        show_output=False
    ).stdout.split("\0")

    for notebook in tracked_files:
        if notebook.endswith(".ipynb"):
            run_git([
                "restore", "--worktree", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Pull/rebase GitHub changes
    # --------------------------------------------------

    pull = subprocess.run(
        ["git", *auth_options, "pull", "--rebase", "origin", branch],
        cwd=REPO_PATH,
        env=git_env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if pull.stdout:
        print(pull.stdout, end="")
    if pull.stderr:
        print(pull.stderr, end="")

    if pull.returncode != 0:
        conflicted = run_git(
            ["diff", "--name-only", "--diff-filter=U"],
            show_output=False
        ).stdout.splitlines()

        if conflicted == [".gitignore"]:
            ours = run_git(
                ["show", ":2:.gitignore"],
                show_output=False
            ).stdout

            theirs = run_git(
                ["show", ":3:.gitignore"],
                show_output=False
            ).stdout

            if ours.strip() != theirs.strip():
                raise RuntimeError(
                    "The .gitignore versions differ. The rebase is paused; "
                    "inspect the conflict before continuing."
                )

            gitignore_path.write_text(
                ours + ("\n" if ours and not ours.endswith("\n") else ""),
                encoding="utf-8"
            )

            run_git(["add", "--", ".gitignore"])
            run_git([
                "-c", "core.editor=true", "rebase", "--continue"
            ])

        else:
            raise RuntimeError(
                "Pull failed. Check the Git output above and `git status`. "
                "If running locally, verify your GitHub authentication."
            )

    # --------------------------------------------------
    # Push synchronized changes
    # --------------------------------------------------

    run_git(
        [*auth_options, "push", "origin", branch],
        env=git_env
    )

    print(
        f"Successfully pushed to {GITHUB_USER}/{GITHUB_REPO} "
        f"on branch '{branch}'."
    )

finally:
    if askpass_dir is not None:
        askpass_dir.cleanup()

    git_env.pop("GITHUB_TOKEN", None)

Runtime: Local Windows
Repository: C:\Users\idosc\Corpus2GeoRAG
Branch: main
Authentication: local Git credentials.
Excluded oversized file: ApplyNER/NER_output/model_evaluation/fine_tuning/flert_5label/FlertDate/best-model.pt (2152.35 MiB). Kept on disk.
Excluded oversized file: ApplyNER/NER_output/model_evaluation/fine_tuning/flert_5label/FlertDate/final-model.pt (2152.35 MiB). Kept on disk.
Excluded oversized file: ApplyNER/NER_output/model_evaluation/fine_tuning/flert_5label/original_flert_3epochs/best-model.pt (2152.35 MiB). Kept on disk.
Excluded oversized file: ApplyNER/NER_output/model_evaluation/fine_tuning/flert_5label/original_flert_3epochs/final-model.pt (2152.35 MiB). Kept on disk.
Excluded oversized file: ApplyNER/NER_output/model_evaluation/fine_tuning/flert_stage1/best-model.pt (2152.33 MiB). Kept on disk.
Excluded oversized file: ApplyNER/NER_output/model_evaluation/fine_tuning/flert_stage1/final-model.pt (2152.33 MiB). Kept on disk.
Excluded oversized file: ApplyNER/N